# 06b — Amazon Recall@K and NDCG@K

**input:** `data/amazon/items.pkl`, `data/amazon/embeddings.npy`, `data/amazon/results_ours.pkl`

Same metrics as notebook 06 — applied to Amazon Clothing.

In [ ]:
import numpy as np
import pandas as pd
import pickle, sys
from pathlib import Path
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import matplotlib.pyplot as plt

sys.path.append(str(Path('../')))
from src.query_templates_amazon import get_query_for_item

DATA_DIR    = Path('../data/amazon')
RESULTS_DIR = Path('../results')

items      = pd.read_pickle(DATA_DIR / 'items.pkl')
embeddings = np.load(DATA_DIR / 'embeddings.npy')
sbert      = SentenceTransformer('all-MiniLM-L6-v2')

with open(DATA_DIR / 'results_ours.pkl', 'rb') as f:
    ours = pickle.load(f)
with open(DATA_DIR / 'results_random.pkl', 'rb') as f:
    rand = pickle.load(f)
with open(DATA_DIR / 'results_taxonomy.pkl', 'rb') as f:
    taxo = pickle.load(f)

df_ours     = ours['df']
df_random   = rand['df']
df_taxonomy = taxo['df']
target_items = ours['target_items']

STOP_SIZE = 5
MAX_TURNS = 6
TOP_N     = 500
np.random.seed(42)
print('imports OK')

ModuleNotFoundError: No module named 'query_templates_amazon'

### Helper functions

In [ ]:
import sys
sys.path.insert(0, '../src')
from core import (
    entropy,
    adaptive_k,
    simulated_user,
    cluster_candidates,
    retrieve_candidates,
)
def make_query_embedding(target_idx):
    cat = items.iloc[target_idx]['category']
    q   = get_query_for_item(cat)
    return sbert.encode(q, normalize_embeddings=True)

def random_partition(C, k):
    s = C.copy(); np.random.shuffle(s)
    return [list(c) for c in np.array_split(s, k) if len(c) > 0]

item_primary_cat = items['category'].str.split('>').str[-1].str.strip().values
def taxonomy_partition(C):
    groups = {}
    for idx in C: groups.setdefault(item_primary_cat[idx], []).append(idx)
    p = [g for g in groups.values() if g]
    return sorted(p, key=len, reverse=True)[:6] if len(p) > 6 else p

def rank_target(C, target_idx, q_emb):
    if not C: return len(embeddings) + 1
    scores = embeddings[C] @ q_emb
    ranked = [C[i] for i in np.argsort(scores)[::-1]]
    return ranked.index(target_idx) + 1 if target_idx in ranked else len(ranked) + 1

def recall_at_k(ranks, k): return np.mean([1 if r <= k else 0 for r in ranks])
def ndcg_at_k(ranks, k):
    return np.mean([1/np.log2(r+1) if r <= k else 0 for r in ranks]) / (1/np.log2(2))

print('functions OK')

### Run and compute ranks

In [ ]:
def run_and_rank(target_idx, method='ours'):
    q_emb = make_query_embedding(target_idx)
    C = retrieve(q_emb, TOP_N)
    if target_idx not in C: C.append(target_idx)
    turn = 0
    while len(C) > STOP_SIZE and turn < MAX_TURNS:
        if method == 'ours':
            p = cluster_candidates(C, adaptive_k(len(C)))
        elif method == 'random':
            p = random_partition(C, adaptive_k(len(C)))
        elif method == 'taxonomy':
            p = taxonomy_partition(C)
            if len(p) <= 1: break
        C = p[simulated_user(p, target_idx)]
        turn += 1
    return rank_target(C, target_idx, q_emb)

ranks_ours, ranks_random, ranks_taxonomy = [], [], []
for t in tqdm(target_items, desc='Computing ranks'):
    ranks_ours.append(run_and_rank(int(t), 'ours'))
    ranks_random.append(run_and_rank(int(t), 'random'))
    ranks_taxonomy.append(run_and_rank(int(t), 'taxonomy'))
print('Done.')

### Results

In [ ]:
ks = [1, 3, 5, 10]

print('── Amazon Recall@K ──────────────────────────────────')
print(f'{"K":>4}  {"Semantic":>10}  {"Taxonomy":>10}  {"Random":>10}')
print('-' * 40)
for k in ks:
    print(f'{k:>4}  {recall_at_k(ranks_ours,k):>10.3f}  {recall_at_k(ranks_taxonomy,k):>10.3f}  {recall_at_k(ranks_random,k):>10.3f}')

print()
print('── Amazon NDCG@K ────────────────────────────────────')
print(f'{"K":>4}  {"Semantic":>10}  {"Taxonomy":>10}  {"Random":>10}')
print('-' * 40)
for k in ks:
    print(f'{k:>4}  {ndcg_at_k(ranks_ours,k):>10.3f}  {ndcg_at_k(ranks_taxonomy,k):>10.3f}  {ndcg_at_k(ranks_random,k):>10.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, fn, title in [(axes[0], recall_at_k, 'Recall@K'), (axes[1], ndcg_at_k, 'NDCG@K')]:
    ax.plot(ks, [fn(ranks_ours,k)     for k in ks], marker='o', color='steelblue', label='Semantic (ours)', linewidth=2)
    ax.plot(ks, [fn(ranks_taxonomy,k) for k in ks], marker='s', color='coral',     label='Taxonomy', linestyle='--')
    ax.plot(ks, [fn(ranks_random,k)   for k in ks], marker='^', color='gray',      label='Random',   linestyle=':')
    ax.set_xlabel('K'); ax.set_title(f'Amazon — {title} (higher = better)')
    ax.legend(); ax.grid(alpha=0.3); ax.set_xticks(ks)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'amazon_recall_ndcg.png', dpi=150)
plt.show()
print('Saved: results/amazon_recall_ndcg.png')
print()
print('✅ notebook 06b complete')